# Week 77, Day 3: REST design

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w077d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('miniflask.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of Flask that runs in the page (Week 77).\n\nThe names and behaviour follow Flask\'s own, so code written against this module also runs with\nthe real Flask (pip install flask): Flask, request, g, jsonify, abort, redirect, make_response,\nurl_for, Blueprint, and app.test_client(). It\'s a teaching copy, not the real thing: no templates,\nsessions, streaming or debugger.\n"""\nimport io\nimport json\nimport re\nfrom contextvars import ContextVar\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qs, urlencode, urlsplit\n\n_current = ContextVar("request")\n\n\nclass HTTPException(Exception):\n    def __init__(self, code, description=None):\n        self.code = code\n        self.name = HTTPStatus(code).phrase\n        self.description = description or self.name\n        super().__init__(f"{code} {self.name}: {self.description}")\n\n\ndef abort(code, description=None):\n    raise HTTPException(code, description)\n\n\nclass Headers:\n    """Case-insensitive headers."""\n    def __init__(self, items=()):\n        self._items = [(k, str(v)) for k, v in (items.items() if isinstance(items, dict) else items)]\n\n    def get(self, name, default=None):\n        for k, v in self._items:\n            if k.lower() == name.lower():\n                return v\n        return default\n\n    def __getitem__(self, name):\n        value = self.get(name)\n        if value is None:\n            raise KeyError(name)\n        return value\n\n    def __contains__(self, name):\n        return self.get(name) is not None\n\n    def __setitem__(self, name, value):\n        self._items = [(k, v) for k, v in self._items if k.lower() != name.lower()] + [(name, str(value))]\n\n    def items(self):\n        return list(self._items)\n\n\nclass Args(dict):\n    """Query or form values: .get(name) gives the first value; .getlist(name) all of them."""\n    def __init__(self, pairs):\n        super().__init__()\n        self._lists = {}\n        for k, v in pairs:\n            self._lists.setdefault(k, []).append(v)\n            dict.setdefault(self, k, v)\n\n    def get(self, key, default=None, type=None):\n        if key not in self:\n            return default\n        value = self[key]\n        if type is not None:\n            try:\n                return type(value)\n            except (TypeError, ValueError):\n                return default\n        return value\n\n    def getlist(self, key):\n        return list(self._lists.get(key, []))\n\n\nclass Request:\n    def __init__(self, environ):\n        self.environ = environ\n        self.method = environ["REQUEST_METHOD"].upper()\n        self.path = environ.get("PATH_INFO", "/") or "/"\n        self.args = Args(_pairs(environ.get("QUERY_STRING", "")))\n        items = [(k[5:].replace("_", "-").title(), v) for k, v in environ.items() if k.startswith("HTTP_")]\n        if environ.get("CONTENT_TYPE"):\n            items.append(("Content-Type", environ["CONTENT_TYPE"]))\n        self.headers = Headers(items)\n        length = int(environ.get("CONTENT_LENGTH") or 0)\n        self.data = environ["wsgi.input"].read(length) if length else b""\n        self.content_type = environ.get("CONTENT_TYPE", "")\n        self.form = Args(_pairs(self.data.decode("utf-8"))) if self.content_type.startswith("application/x-www-form-urlencoded") else Args([])\n        self.view_args = {}\n        cookie = environ.get("HTTP_COOKIE", "")\n        self.cookies = dict(p.strip().split("=", 1) for p in cookie.split(";") if "=" in p)\n\n    @property\n    def is_json(self):\n        return self.content_type.split(";")[0].strip() == "application/json"\n\n    def get_json(self, silent=False, force=False):\n        if not (self.is_json or force):\n            if silent:\n                return None\n            abort(415, "expected a JSON body (Content-Type: application/json)")\n        try:\n            return json.loads(self.data or b"null")\n        except ValueError:\n            if silent:\n                return None\n            abort(400, "the body isn\'t valid JSON")\n\n    @property\n    def json(self):\n        return self.get_json()\n\n\ndef _pairs(text):\n    return [(k, v) for k, vs in parse_qs(text, keep_blank_values=True).items() for v in vs]\n\n\nclass _Proxy:\n    def __init__(self, getter):\n        object.__setattr__(self, "_getter", getter)\n\n    def __getattr__(self, name):\n        return getattr(self._getter(), name)\n\n    def __setattr__(self, name, value):\n        setattr(self._getter(), name, value)\n\n\ndef _request():\n    try:\n        return _current.get()[0]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\ndef _g():\n    try:\n        return _current.get()[1]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\nclass _G:\n    def get(self, name, default=None):\n        return self.__dict__.get(name, default)\n\n\nrequest = _Proxy(_request)\ng = _Proxy(_g)\n\n\nclass Response:\n    def __init__(self, response=b"", status=200, headers=None, mimetype=None, content_type=None):\n        if isinstance(response, str):\n            response = response.encode("utf-8")\n        self.data = response\n        self.status_code = status\n        self.headers = Headers(headers or {})\n        if content_type or mimetype or "Content-Type" not in self.headers:\n            ctype = content_type or (f"{mimetype}; charset=utf-8" if mimetype and mimetype.startswith("text/") else mimetype) or "text/html; charset=utf-8"\n            self.headers["Content-Type"] = ctype\n\n    @property\n    def status(self):\n        return f"{self.status_code} {HTTPStatus(self.status_code).phrase}"\n\n    @property\n    def text(self):\n        return self.data.decode("utf-8")\n\n    def get_json(self):\n        return json.loads(self.data)\n\n    @property\n    def json(self):\n        return self.get_json()\n\n    def set_cookie(self, key, value="", max_age=None, path="/", secure=False, httponly=False, samesite=None):\n        parts = [f"{key}={value}"]\n        if max_age is not None:\n            parts.append(f"Max-Age={max_age}")\n        parts.append(f"Path={path}")\n        if secure:\n            parts.append("Secure")\n        if httponly:\n            parts.append("HttpOnly")\n        if samesite:\n            parts.append(f"SameSite={samesite}")\n        self.headers._items.append(("Set-Cookie", "; ".join(parts)))\n\n\ndef jsonify(*args, **kwargs):\n    data = args[0] if len(args) == 1 else (list(args) if args else kwargs)\n    return Response(json.dumps(data), headers={"Content-Type": "application/json"})\n\n\ndef redirect(location, code=302):\n    return Response(b"", code, {"Location": location})\n\n\ndef make_response(*args):\n    rv = args[0] if len(args) == 1 else args\n    return _App._to_response(rv)\n\n\ndef url_for(endpoint, **values):\n    return _current.get()[2].url_for(endpoint, **values)\n\n\n_CONVERTERS = {"int": (r"\\d+", int), "float": (r"\\d+(?:\\.\\d+)?", float), "path": (r".+", str), "string": (r"[^/]+", str)}\n\n\nclass _Rule:\n    def __init__(self, rule, endpoint, view, methods):\n        self.rule, self.endpoint, self.view = rule, endpoint, view\n        self.methods = {m.upper() for m in methods} | ({"HEAD"} if "GET" in {m.upper() for m in methods} else set())\n        regex, self.converters = "", {}\n        for piece in re.split(r"(<[^>]+>)", rule):\n            m = re.fullmatch(r"<(?:(\\w+):)?(\\w+)>", piece)\n            if m:\n                kind, name = m.group(1) or "string", m.group(2)\n                pattern, convert = _CONVERTERS[kind]\n                regex += f"(?P<{name}>{pattern})"\n                self.converters[name] = convert\n            else:\n                regex += re.escape(piece)\n        self.regex = re.compile(regex)\n\n    def match(self, path):\n        m = self.regex.fullmatch(path)\n        if not m:\n            return None\n        return {k: self.converters[k](v) for k, v in m.groupdict().items()}\n\n    def build(self, values):\n        def fill(m):\n            return str(values.pop(m.group(2)))\n        path = re.sub(r"<(?:(\\w+):)?(\\w+)>", fill, self.rule)\n        return path + (f"?{urlencode(values)}" if values else "")\n\n\nclass _Setup:\n    def route(self, rule, methods=("GET",), endpoint=None):\n        def decorate(view):\n            self._add(rule, endpoint or view.__name__, view, methods)\n            return view\n        return decorate\n\n    def get(self, rule, **kw):\n        return self.route(rule, methods=["GET"], **kw)\n\n    def post(self, rule, **kw):\n        return self.route(rule, methods=["POST"], **kw)\n\n    def put(self, rule, **kw):\n        return self.route(rule, methods=["PUT"], **kw)\n\n    def patch(self, rule, **kw):\n        return self.route(rule, methods=["PATCH"], **kw)\n\n    def delete(self, rule, **kw):\n        return self.route(rule, methods=["DELETE"], **kw)\n\n\nclass Blueprint(_Setup):\n    def __init__(self, name, import_name, url_prefix=""):\n        self.name, self.url_prefix = name, url_prefix\n        self.deferred = []\n\n    def _add(self, rule, endpoint, view, methods):\n        self.deferred.append((rule, endpoint, view, methods))\n\n\nclass _App(_Setup):\n    def __init__(self, import_name):\n        self.import_name = import_name\n        self.config = {"TESTING": False}\n        self.rules = []\n        self.before_funcs, self.after_funcs, self.error_handlers = [], [], {}\n\n    def _add(self, rule, endpoint, view, methods):\n        self.rules.append(_Rule(rule, endpoint, view, methods))\n\n    def register_blueprint(self, bp, url_prefix=None):\n        prefix = url_prefix if url_prefix is not None else bp.url_prefix\n        for rule, endpoint, view, methods in bp.deferred:\n            self._add(prefix + rule, f"{bp.name}.{endpoint}", view, methods)\n\n    def before_request(self, f):\n        self.before_funcs.append(f)\n        return f\n\n    def after_request(self, f):\n        self.after_funcs.append(f)\n        return f\n\n    def errorhandler(self, code):\n        def decorate(f):\n            self.error_handlers[code] = f\n            return f\n        return decorate\n\n    def url_for(self, endpoint, **values):\n        for r in self.rules:\n            if r.endpoint == endpoint:\n                return r.build(dict(values))\n        raise LookupError(f"no endpoint {endpoint!r}")\n\n    @staticmethod\n    def _to_response(rv):\n        status, headers = None, None\n        if isinstance(rv, tuple):\n            if len(rv) == 3:\n                rv, status, headers = rv\n            elif isinstance(rv[1], (dict, list)):\n                rv, headers = rv\n            else:\n                rv, status = rv\n        if isinstance(rv, Response):\n            response = rv\n        elif isinstance(rv, (dict, list)):\n            response = jsonify(rv)\n        elif isinstance(rv, (str, bytes)):\n            response = Response(rv)\n        elif rv is None:\n            raise TypeError("the view function did not return a valid response")\n        else:\n            raise TypeError(f"can\'t turn {type(rv).__name__} into a response")\n        if status is not None:\n            response.status_code = status\n        for k, v in (headers.items() if isinstance(headers, dict) else headers or []):\n            response.headers[k] = v\n        return response\n\n    def _error(self, e):\n        handler = self.error_handlers.get(e.code)\n        if handler:\n            return self._to_response(handler(e))\n        body = f"<h1>{e.code} {e.name}</h1><p>{e.description}</p>"\n        return Response(body, e.code)\n\n    def _dispatch(self, req):\n        path = req.path\n        allowed = set()\n        for r in self.rules:\n            values = r.match(path)\n            if values is None:\n                continue\n            if req.method in r.methods:\n                req.view_args = values\n                return r.view(**values)\n            allowed |= r.methods\n        if allowed:\n            raise _MethodNotAllowed(sorted(allowed | {"OPTIONS"}))\n        abort(404, "The requested URL was not found on the server.")\n\n    def full_dispatch(self, environ):\n        req = Request(environ)\n        token = _current.set((req, _G(), self))\n        try:\n            try:\n                rv = None\n                for f in self.before_funcs:\n                    rv = f()\n                    if rv is not None:\n                        break\n                if rv is None:\n                    rv = self._dispatch(req)\n                response = self._to_response(rv)\n            except _MethodNotAllowed as e:\n                response = self._error(HTTPException(405))\n                response.headers["Allow"] = ", ".join(e.allowed)\n            except HTTPException as e:\n                response = self._error(e)\n            except Exception as e:\n                if self.config.get("TESTING") and 500 not in self.error_handlers:\n                    raise\n                response = self._error(HTTPException(500)) if 500 not in self.error_handlers else self._to_response(self.error_handlers[500](e))\n                response.status_code = 500\n            for f in self.after_funcs:\n                response = f(response) or response\n            if req.method == "HEAD":\n                response.headers["Content-Length"] = len(response.data)\n                response.data = b""\n            return response\n        finally:\n            _current.reset(token)\n\n    def __call__(self, environ, start_response):\n        response = self.full_dispatch(environ)\n        if "Content-Length" not in response.headers:\n            response.headers["Content-Length"] = len(response.data)\n        start_response(response.status, response.headers.items())\n        return [response.data]\n\n    def test_client(self):\n        return _TestClient(self)\n\n\nclass _MethodNotAllowed(Exception):\n    def __init__(self, allowed):\n        self.allowed = allowed\n\n\nclass Flask(_App):\n    pass\n\n\nclass _TestClient:\n    def __init__(self, app):\n        self.app = app\n        self.cookies = {}\n\n    def open(self, path, method="GET", json=None, data=None, headers=None, query_string=None):\n        url = urlsplit(path)\n        query = url.query\n        if query_string:\n            query = (query + "&" if query else "") + urlencode(query_string, doseq=True)\n        body, ctype = b"", None\n        if json is not None:\n            import json as _json\n            body, ctype = _json.dumps(json).encode("utf-8"), "application/json"\n        elif isinstance(data, dict):\n            body, ctype = urlencode(data, doseq=True).encode("utf-8"), "application/x-www-form-urlencoded"\n        elif data is not None:\n            body = data.encode("utf-8") if isinstance(data, str) else data\n        environ = {"REQUEST_METHOD": method.upper(), "PATH_INFO": url.path or "/", "QUERY_STRING": query,\n                   "SERVER_NAME": "localhost", "SERVER_PORT": "80", "wsgi.url_scheme": "http",\n                   "wsgi.input": io.BytesIO(body), "CONTENT_LENGTH": str(len(body))}\n        headers = dict(headers or {})\n        if ctype and not any(k.lower() == "content-type" for k in headers):\n            environ["CONTENT_TYPE"] = ctype\n        for k, v in headers.items():\n            if k.lower() == "content-type":\n                environ["CONTENT_TYPE"] = v\n            else:\n                environ["HTTP_" + k.upper().replace("-", "_")] = v\n        if self.cookies:\n            environ["HTTP_COOKIE"] = "; ".join(f"{k}={v}" for k, v in self.cookies.items())\n        response = self.app.full_dispatch(environ)\n        for k, v in response.headers.items():\n            if k.lower() == "set-cookie":\n                name, _, rest = v.partition("=")\n                self.cookies[name] = rest.split(";")[0]\n        return response\n\n    def get(self, path, **kw):\n        return self.open(path, "GET", **kw)\n\n    def post(self, path, **kw):\n        return self.open(path, "POST", **kw)\n\n    def put(self, path, **kw):\n        return self.open(path, "PUT", **kw)\n\n    def patch(self, path, **kw):\n        return self.open(path, "PATCH", **kw)\n\n    def delete(self, path, **kw):\n        return self.open(path, "DELETE", **kw)\n')

**Predict before you run:** what will this print?



In [ ]:
routes = [
    ("GET", "/getAllOrders"),
    ("POST", "/createOrder"),
    ("GET", "/orders"),
    ("POST", "/orders"),
    ("GET", "/orders/7"),
    ("PATCH", "/orders/7"),
    ("DELETE", "/orders/7"),
    ("POST", "/orders/7/delete"),
]
VERBS = ("get", "create", "delete", "update", "add", "remove")
for method, path in routes:
    verby = any(segment.lower().startswith(VERBS) for segment in path.strip("/").split("/"))
    print(f"{method:<7}{path:<20}{'verb in the path' if verby else 'ok'}")

In [ ]:
import hashlib, json

def etag(resource):
    return '"' + hashlib.sha256(json.dumps(resource, sort_keys=True).encode()).hexdigest()[:12] + '"'

order = {"id": 7, "status": "new", "items": 2}
seen_by_ada = seen_by_bob = etag(order)

def update(changes, if_match):
    global order
    if if_match is None:
        return 428, "send If-Match"
    if if_match != etag(order):
        return 412, "it changed since you read it"
    order = {**order, **changes}
    return 200, etag(order)

print(update({"status": "paid"}, seen_by_ada))
print(update({"items": 3}, seen_by_bob))
print(update({"items": 3}, None), order)

> **Key idea.** REST URLs name resources (plural nouns, lower case, nested one level), never actions; methods say what to do: GET to list or read, POST to create (201 and Location), PUT to replace, PATCH to change part, DELETE (204); awkward actions become resources or state changes. Status codes must be honest (no 200 with an error inside). Keep representations consistent (one field-naming style, ISO dates, money as integers). Return errors as problem details (application/problem+json with type, title, status, detail, instance). Idempotency-Key headers make POST retries safe by replaying the stored response; ETags with If-Match prevent lost updates (412, or 428 when it's missing). Paginate with limited, opaque, validated cursors. Additive changes are safe; breaking changes need a new version and a deprecation period; describe the API with OpenAPI.

### Exercise: A REST route linter

Write `lint(routes)`: `routes` is a list of `(method, path)` pairs. Return a sorted list of `(method, path, problem)` for: `"verb in path"` (a segment that starts with `get`, `create`, `update`, `delete`, `add`, `remove`, `fetch` or `set`, in any case); `"not lower case"` (a segment with a capital letter); `"use hyphens"` (a segment with `_`); `"trailing slash"`; `"file extension"` (a segment ending in `.json`, `.xml` or `.php`); `"too deeply nested"` (more than four segments, counting `{id}`-style parameters); and `"DELETE on a collection"` for a `DELETE` whose last segment isn't a `{parameter}`. A path can have several problems.

In [ ]:
import re

VERBS = ("get", "create", "update", "delete", "add", "remove", "fetch", "set")

def lint(routes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lint():\n    \"REST naming rules\"\n    routes = [(\"GET\", \"/orders\"), (\"GET\", \"/orders/{order_id}\"), (\"POST\", \"/createOrder\"), (\"GET\", \"/order_items/\"),\n              (\"DELETE\", \"/orders\"), (\"GET\", \"/report.json\"), (\"GET\", \"/a/{b}/c/{d}/e\"), (\"GET\", \"/customers/{id}/orders\")]\n    assert lint(routes) == [\n        (\"DELETE\", \"/orders\", \"DELETE on a collection\"),\n        (\"GET\", \"/a/{b}/c/{d}/e\", \"too deeply nested\"),\n        (\"GET\", \"/order_items/\", \"trailing slash\"),\n        (\"GET\", \"/order_items/\", \"use hyphens\"),\n        (\"GET\", \"/report.json\", \"file extension\"),\n        (\"POST\", \"/createOrder\", \"not lower case\"),\n        (\"POST\", \"/createOrder\", \"verb in path\"),\n    ]")

### Exercise: Problem details

Write `problem(status, detail=None, type="about:blank", title=None, instance=None, **extra)`: a problem-details dict with `type`, `title` (the status's phrase from `HTTPStatus` if not given), `status`, and `detail` and `instance` only if given, plus any extra fields. Then `validation_problem(errors, instance=None)`: a `422` problem with the type `"https://api.example.com/problems/validation"`, the title `"Your request has invalid fields"`, the detail `"N field(s) are invalid"`, and `errors`: a list of `{"field": DOTTED.LOC, "message": MSG}` from FastAPI-style errors (the first `loc` entry, such as `body`, is left out). Finally `as_response(problem)`: `(status, headers, body_bytes)` with `Content-Type: application/problem+json`.

In [ ]:
import json
from http import HTTPStatus

def problem(status, detail=None, type="about:blank", title=None, instance=None, **extra):
    ___

def validation_problem(errors, instance=None):
    ___

def as_response(problem_dict):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_problems():\n    \"RFC 9457\"\n    import json\n    assert problem(404, \"no order 7\", instance=\"/orders/7\") == {\"type\": \"about:blank\", \"title\": \"Not Found\", \"status\": 404, \"detail\": \"no order 7\", \"instance\": \"/orders/7\"}\n    assert problem(409, type=\"https://x/problems/stock\", title=\"Not enough stock\", available=2) == {\"type\": \"https://x/problems/stock\", \"title\": \"Not enough stock\", \"status\": 409, \"available\": 2}\n    errors = [{\"loc\": [\"body\", \"title\"], \"msg\": \"Field required\", \"type\": \"missing\"}, {\"loc\": [\"body\", \"lines\", 1, \"qty\"], \"msg\": \"too small\", \"type\": \"x\"}]\n    p = validation_problem(errors, \"/orders\")\n    assert p[\"status\"] == 422 and p[\"detail\"] == \"2 field(s) are invalid\" and p[\"errors\"] == [{\"field\": \"title\", \"message\": \"Field required\"}, {\"field\": \"lines.1.qty\", \"message\": \"too small\"}]\n    status, headers, body = as_response(p)\n    assert status == 422 and headers == {\"Content-Type\": \"application/problem+json\"} and json.loads(body) == p")

### Exercise: Idempotency keys

Write a class `IdempotencyStore(clock, ttl=86400)` with `run(key, body, action)`: `body` is the request's body (bytes) and `action()` does the work and returns `(status, response_body)`. With `key` `None`, just call `action`. Otherwise, the first time a key is seen, call `action`, store `(fingerprint, status, response_body, time)` where the fingerprint is the SHA-256 hex digest of the body, and return the result with `replayed` `False`; when the same key comes back within `ttl` seconds with the **same** fingerprint, return the stored result **without** calling `action`, with `replayed` `True`; with a **different** body, return `(422, b'{"error": "key reused with a different body"}', False)` without calling `action`. Expired keys are forgotten. Return values are `(status, body, replayed)`. Don't store results with a `5xx` status (so the client can retry them).

In [ ]:
import hashlib

class IdempotencyStore:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_idempotency():\n    \"replaying instead of repeating\"\n    now = [0]\n    store = IdempotencyStore(lambda: now[0], ttl=100)\n    created = []\n    def create():\n        created.append(1)\n        return 201, b'{\"id\": %d}' % len(created)\n    assert store.run(\"k1\", b'{\"qty\": 2}', create) == (201, b'{\"id\": 1}', False)\n    assert store.run(\"k1\", b'{\"qty\": 2}', create) == (201, b'{\"id\": 1}', True) and len(created) == 1\n    assert store.run(\"k1\", b'{\"qty\": 3}', create)[0] == 422 and len(created) == 1\n    assert store.run(\"k2\", b'{\"qty\": 2}', create) == (201, b'{\"id\": 2}', False)\n    assert store.run(None, b\"\", create)[2] is False and len(created) == 3\n    now[0] = 101\n    assert store.run(\"k1\", b'{\"qty\": 2}', create) == (201, b'{\"id\": 4}', False)\n    flaky = iter([(503, b\"busy\"), (201, b\"ok\")])\n    assert store.run(\"k3\", b\"x\", lambda: next(flaky)) == (503, b\"busy\", False)\n    assert store.run(\"k3\", b\"x\", lambda: next(flaky)) == (201, b\"ok\", False)")

### Exercise: Conditional updates

Write `etag(resource)`: a quoted string holding the first 16 hex characters of the SHA-256 of the resource as JSON with sorted keys. Then a class `Documents` holding resources by id, with `get(doc_id, if_none_match=None)` returning `(status, etag, body)`: `404` (with `None`s) for a missing id, `304` (with the etag and a `None` body) if `if_none_match` equals the current etag, else `200`; and `update(doc_id, changes, if_match)` returning `(status, etag)`: `404` for a missing id, `428` if `if_match` is `None`, `412` (with the **current** etag) if it doesn't match, else apply the changes and return `200` with the new etag.

In [ ]:
import hashlib, json

def etag(resource):
    ___

class Documents:
    def __init__(self, docs):
        self.docs = docs

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ifmatch():\n    \"If-Match and If-None-Match\"\n    docs = Documents({1: {\"title\": \"Plan\", \"body\": \"v1\"}})\n    status, tag, body = docs.get(1)\n    assert status == 200 and body == {\"title\": \"Plan\", \"body\": \"v1\"} and tag.startswith('\"') and len(tag) == 18\n    assert tag == etag({\"body\": \"v1\", \"title\": \"Plan\"})\n    assert docs.get(1, if_none_match=tag) == (304, tag, None) and docs.get(2) == (404, None, None)\n    assert docs.update(1, {\"body\": \"v2\"}, None)[0] == 428\n    status, new_tag = docs.update(1, {\"body\": \"v2\"}, tag)\n    assert status == 200 and new_tag != tag\n    assert docs.update(1, {\"body\": \"v3\"}, tag) == (412, new_tag) and docs.docs[1][\"body\"] == \"v2\"\n    assert docs.update(9, {}, tag)[0] == 404")

### Exercise: Opaque cursors

Write `encode_cursor(last_id)`: URL-safe base64 (without `=` padding) of the JSON `{"after": last_id}`. Then `decode_cursor(text)`: the `after` id, raising `ValueError("invalid cursor")` for anything that isn't valid base64, valid JSON, a dict with an integer `after` of at least 0. Finally `page(items, cursor=None, limit=10)`: items (dicts with increasing integer `id`s) after the cursor's id, at most `limit` of them, as `{"items": [...], "next": CURSOR or None}`.

In [ ]:
import base64, json

def encode_cursor(last_id):
    ___

def decode_cursor(text):
    ___

def page(items, cursor=None, limit=10):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cursors():\n    \"opaque and validated\"\n    c = encode_cursor(42)\n    assert \"=\" not in c and decode_cursor(c) == 42 and \"after\" not in c\n    for bad in [\"!!!\", \"bm9wZQ\", encode_cursor(42)[:-3], base64.urlsafe_b64encode(b'{\"after\": -1}').decode().rstrip(\"=\"), base64.urlsafe_b64encode(b\"[1]\").decode()]:\n        try:\n            decode_cursor(bad)\n            assert False, bad\n        except ValueError as e:\n            assert str(e) == \"invalid cursor\"\n    items = [{\"id\": i} for i in range(1, 8)]\n    first = page(items, limit=3)\n    assert [i[\"id\"] for i in first[\"items\"]] == [1, 2, 3] and decode_cursor(first[\"next\"]) == 3\n    second = page(items, first[\"next\"], limit=3)\n    third = page(items, second[\"next\"], limit=3)\n    assert [i[\"id\"] for i in third[\"items\"]] == [7] and third[\"next\"] is None")

### Exercise: Is this change breaking?

An API's shape is described as a dict from `"METHOD /path"` to `{"params": {name: required_bool}, "response": {field: type_name}}`. Write `compare(old, new)`: a sorted list of `(kind, endpoint, description)` where `kind` is `"breaking"` or `"safe"`: a removed endpoint is breaking (`"endpoint removed"`), an added one safe (`"endpoint added"`); for endpoints in both, a removed response field is breaking (`"field F removed"`), a changed type breaking (`"field F changed from A to B"`), an added field safe (`"field F added"`); a new **required** parameter is breaking (`"required parameter P added"`), a new optional one safe (`"optional parameter P added"`), an optional parameter becoming required breaking (`"parameter P now required"`), and a removed parameter safe (`"parameter P removed"`). Then `needs_new_version(old, new)`.

In [ ]:
def compare(old, new):
    ___

def needs_new_version(old, new):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_compat():\n    \"additive versus breaking changes\"\n    old = {\n        \"GET /orders\": {\"params\": {\"status\": False}, \"response\": {\"id\": \"int\", \"total\": \"int\", \"placed\": \"str\"}},\n        \"GET /orders/{id}\": {\"params\": {}, \"response\": {\"id\": \"int\"}},\n        \"POST /orders\": {\"params\": {}, \"response\": {\"id\": \"int\"}},\n    }\n    new = {\n        \"GET /orders\": {\"params\": {\"status\": True, \"limit\": False}, \"response\": {\"id\": \"int\", \"total\": \"str\", \"currency\": \"str\"}},\n        \"GET /orders/{id}\": {\"params\": {\"expand\": True}, \"response\": {\"id\": \"int\"}},\n        \"GET /customers\": {\"params\": {}, \"response\": {\"id\": \"int\"}},\n    }\n    assert compare(old, new) == [\n        (\"breaking\", \"GET /orders\", \"field placed removed\"),\n        (\"breaking\", \"GET /orders\", \"field total changed from int to str\"),\n        (\"breaking\", \"GET /orders\", \"parameter status now required\"),\n        (\"breaking\", \"GET /orders/{id}\", \"required parameter expand added\"),\n        (\"breaking\", \"POST /orders\", \"endpoint removed\"),\n        (\"safe\", \"GET /customers\", \"endpoint added\"),\n        (\"safe\", \"GET /orders\", \"field currency added\"),\n        (\"safe\", \"GET /orders\", \"optional parameter limit added\"),\n    ]\n    assert needs_new_version(old, new) and not needs_new_version(old, {**old, \"GET /x\": {\"params\": {}, \"response\": {}}})")

### Bug hunt: An error disguised as success

The endpoint reports a missing order with status `200` and an error in the body, so the client's status check thinks it worked and this prints `200 ok?`. Return a real `404`, so it prints `404 no order 9`.

In [ ]:
from miniflask import Flask

app = Flask(__name__)
ORDERS = {7: {"id": 7}}

@app.get("/orders/<int:order_id>")
def get_order(order_id):
    if order_id not in ORDERS:
        return {"error": f"no order {order_id}"}
    return ORDERS[order_id]

r = app.test_client().get("/orders/9")
print(r.status_code, "ok?" if r.status_code == 200 else r.get_json()["error"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"honest status codes\"\n    expect_output(\"404 no order 9\")")

### Bug hunt: A PUT that wasn't idempotent

`PUT` should **replace** the resource, so sending it twice leaves the same result, but this handler appends the tags, so the retry doubles them and this prints `['red', 'blue', 'red', 'blue']`. Replace instead, so it prints `['red', 'blue']`.

In [ ]:
from miniflask import Flask, request

app = Flask(__name__)
PRODUCTS = {1: {"tags": []}}

@app.put("/products/<int:pid>/tags")
def set_tags(pid):
    PRODUCTS[pid]["tags"].extend(request.get_json())
    return {"tags": PRODUCTS[pid]["tags"]}

client = app.test_client()
client.put("/products/1/tags", json=["red", "blue"])
print(client.put("/products/1/tags", json=["red", "blue"]).get_json()["tags"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"PUT replaces\"\n    expect_output(\"['red', 'blue']\")")

### Bug hunt: A cursor that crashed the server

The endpoint decodes whatever cursor the client sends without checking it, so a garbled one raises an exception and the server answers `500`: this prints `500`. Treat a bad cursor as the client's mistake, answering `400`, so this prints `400`.

In [ ]:
import base64, json
from miniflask import Flask, request

app = Flask(__name__)

@app.get("/items")
def items():
    after = 0
    cursor = request.args.get("cursor")
    if cursor:
        after = json.loads(base64.urlsafe_b64decode(cursor + "=="))["after"]
    return {"after": after}

print(app.test_client().get("/items?cursor=garbled!").status_code)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"validate what the client sends\"\n    expect_output(\"400\")")

### Puzzle: Routes for a resource

Arrange the routes for a REST `invoices` resource, from listing to deleting, with sending an invoice modelled as a resource of its own. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the status for a successful DELETE with nothing to return
q = ___   # the status when If-Match doesn't match the current ETag
r = ___   # the media type of an RFC 9457 error body
s = ___   # "breaking" or "safe": adding a new optional field to a response
t = ___   # the header a client sends so a retried POST isn't done twice

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 204 and q == 412 and r == \"application/problem+json\", \"p, q and r\"\n    assert s == \"safe\" and t == \"Idempotency-Key\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a well-behaved resource

Write `create_app(clock=lambda: 0)`: a Flask (miniflask) app with an `orders` resource (orders have `id`, `item` (a non-empty string) and `qty` (an integer from 1 to 100)). Every error is a problem-details response (`Content-Type: application/problem+json`, with `type` `about:blank`, `title`, `status`, `detail` and `instance` set to the request's path). `GET /orders` lists them (by id). `POST /orders` creates one (`201`, `Location`, and an `ETag` header); invalid bodies get `422` (detail `"item must be a non-empty string and qty an integer from 1 to 100"`); and an `Idempotency-Key` header makes retries safe: the same key and body replay the stored response (with the same status, body and `Location`, and a header `Idempotent-Replayed: true`), while the same key with a different body gets `422` with detail `"Idempotency-Key reused with a different body"`. `GET /orders/<id>` returns the order with its `ETag`, or `304` for a matching `If-None-Match`. `PATCH /orders/<id>` requires `If-Match` (`428`, detail `"If-Match is required"`), refuses a stale one (`412`, detail `"the order has changed"`), validates the merged order (`422`), and returns it with its new `ETag`. Missing orders get `404` (detail `"no order N"`).

In [ ]:
import hashlib, json
from miniflask import Flask, request, url_for, Response

def create_app(clock=lambda: 0):
    app = Flask(__name__)
    ___
    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_resource():\n    \"problem details, idempotency and conditional requests\"\n    import json\n    client = create_app().test_client()\n    r = client.post(\"/orders\", json={\"item\": \"tea\", \"qty\": 2}, headers={\"Idempotency-Key\": \"k1\"})\n    assert r.status_code == 201 and r.headers[\"Location\"] == \"/orders/1\" and r.get_json() == {\"id\": 1, \"item\": \"tea\", \"qty\": 2}\n    tag = r.headers[\"ETag\"]\n    again = client.post(\"/orders\", json={\"item\": \"tea\", \"qty\": 2}, headers={\"Idempotency-Key\": \"k1\"})\n    assert again.status_code == 201 and again.get_json()[\"id\"] == 1 and again.headers[\"Idempotent-Replayed\"] == \"true\"\n    assert len(client.get(\"/orders\").get_json()) == 1\n    clash = client.post(\"/orders\", json={\"item\": \"jam\", \"qty\": 1}, headers={\"Idempotency-Key\": \"k1\"})\n    assert clash.status_code == 422 and clash.get_json()[\"detail\"] == \"Idempotency-Key reused with a different body\"\n    assert clash.headers[\"Content-Type\"] == \"application/problem+json\" and clash.get_json()[\"instance\"] == \"/orders\"\n    bad = client.post(\"/orders\", json={\"item\": \"\", \"qty\": 500})\n    assert bad.status_code == 422 and bad.get_json()[\"title\"] in (\"Unprocessable Content\", \"Unprocessable Entity\")\n    assert client.get(\"/orders/1\", headers={\"If-None-Match\": tag}).status_code == 304\n    assert client.patch(\"/orders/1\", json={\"qty\": 3}).status_code == 428\n    r = client.patch(\"/orders/1\", json={\"qty\": 3}, headers={\"If-Match\": tag})\n    assert r.status_code == 200 and r.get_json()[\"qty\"] == 3 and r.headers[\"ETag\"] != tag\n    stale = client.patch(\"/orders/1\", json={\"qty\": 4}, headers={\"If-Match\": tag})\n    assert stale.status_code == 412 and stale.get_json()[\"detail\"] == \"the order has changed\"\n    assert client.patch(\"/orders/1\", json={\"qty\": 0}, headers={\"If-Match\": r.headers[\"ETag\"]}).status_code == 422\n    missing = client.get(\"/orders/9\")\n    assert missing.status_code == 404 and missing.get_json() == {\"type\": \"about:blank\", \"title\": \"Not Found\", \"status\": 404, \"detail\": \"no order 9\", \"instance\": \"/orders/9\"}")

## Warm-up

Half of these routes put the action in the URL, where it duplicates (or contradicts) the method: `POST /orders/7/delete` is a `DELETE` in disguise. In REST the URL names a **thing**, the method says what to do to it, and the whole API becomes guessable.

## Resources and methods

**Resources** are the nouns of an API: orders, customers, products. Each has a URL:

- a **collection**: `/orders`; an **item** in it: `/orders/7`; plural nouns, lower case, with hyphens between words (`/order-items`);
- related things **nested one level**: `/customers/12/orders` (Ada's orders); deeper nesting gets unwieldy, so link instead;
- no verbs, no file extensions (`.json`), no trailing slashes, and query strings for filtering, sorting and paging (`/orders?status=shipped&sort=-placed`).

**Methods** map onto operations:

| Operation | Request | Success |
|---|---|---|
| list | `GET /orders` | `200` with the list |
| read | `GET /orders/7` | `200`; `404` if it doesn't exist |
| create | `POST /orders` | `201` with the new order and `Location: /orders/42` |
| replace | `PUT /orders/7` | `200` (or `201` if it created it) |
| update part | `PATCH /orders/7` | `200` with the result |
| delete | `DELETE /orders/7` | `204`, no body |

Some actions aren't naturally create, read, update or delete: cancelling an order, sending an invoice. Model them as a **resource of their own** (`POST /orders/7/cancellation`, creating a cancellation) or as a state change (`PATCH /orders/7` with `{"status": "cancelled"}`), rather than a verb in the path.

## Responses and errors

**Status codes must tell the truth.** A `200` with `{"error": "not found"}` in the body breaks every client, cache and monitoring tool that trusts the status. Use the codes from Week 76 Day 2: `400` (malformed), `401` (who are you?), `403` (not allowed), `404`, `409` (conflicts with the current state, such as a duplicate), `412` (a precondition failed, below), `422` (well-formed but invalid), `429`, and `5xx` only for the server's own failures.

**Consistent representations.** One naming style for fields everywhere (`snake_case` or `camelCase`, never both), ISO 8601 dates and times with time zones (`2026-11-10T09:30:00Z`), money as integers in the smallest unit (or decimal strings) with a currency, and the same shape for a thing wherever it appears.

**Errors as problem details.** RFC 9457 standardises an error body with the media type `application/problem+json`:

```json
{
  "type": "https://api.example.com/problems/insufficient-stock",
  "title": "Not enough stock",
  "status": 409,
  "detail": "Only 2 left of product 7, but 5 were requested.",
  "instance": "/orders/1042"
}
```

`type` identifies the kind of problem (a URL that documents it; `about:blank` when there's nothing more than the status), `title` is a short summary of that kind, `detail` explains this occurrence, and extra fields (such as a list of validation `errors`) are allowed. Clients can handle every endpoint's errors with one piece of code.

## Safe retries and concurrent updates

**Idempotency keys.** A `POST` that timed out may or may not have worked (Week 76 Day 3). To make it safe to retry, the client sends a unique **`Idempotency-Key`** header (a random UUID) with each new operation; the server stores the response it gave for each key, and when the **same key** arrives again, it returns the **stored response** instead of doing the work twice. If the same key arrives with a **different** body, that's a client bug: refuse it (`422` or `409`). Keys expire after a day or so. Payment APIs work this way.

**Conditional updates.** Two people edit the same order: both read it, both change it, the second save silently overwrites the first (Week 75 Day 5's lost update, over HTTP). The fix is **optimistic concurrency with ETags**: every representation has an `ETag` (a version, often a hash of its content); a client updating it sends **`If-Match: "the-etag-it-read"`**; the server compares it with the current ETag and refuses with **`412 Precondition Failed`** if they differ (someone changed it meanwhile). Servers can require it, answering `428 Precondition Required` to updates without `If-Match`.

## Pagination and versions

**Pagination** (Week 76 Day 3, from the client's side). Offer a `limit` (with a maximum) and either page numbers or, better for large or changing data, **cursors**: an **opaque** string the client hands back to get the next page. Encode the cursor (for instance, base64 of a small JSON object holding the last id seen) so clients can't depend on its insides, and **validate** it when it comes back: a tampered or malformed cursor is a `400`, not a crash. Return the next cursor (or `null`) in the body, or a `Link` header.

**Evolving an API.** Clients you can't update depend on every detail of your responses. **Additive** changes are safe: a new endpoint, a new optional parameter, a new field in a response (clients should ignore fields they don't know). **Breaking** changes are not: removing or renaming a field or endpoint, changing a type, making an optional parameter required, changing what a status code means. Make breaking changes in a new **version** (`/v2/orders`, or a version header) while keeping the old one running for a published deprecation period, announced with headers such as `Deprecation` and `Sunset`.

**Describe it.** An **OpenAPI** document (FastAPI writes one for you at `/openapi.json`) lists every endpoint, parameter, body and response, and drives documentation, client generators and contract tests.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the documentation of a public API you've used (GitHub's or Stripe's are good examples) and find its idempotency, pagination and error conventions. Tomorrow: **authentication**: storing passwords safely, sessions and tokens, and deciding who may do what.